In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
!pip install scipy

In [ ]:
df = pd.read_csv('comet_v3.csv') # a dump of Comet runs

In [ ]:
cols_to_plot = [
    'sweep_pair.num_heads',
    'sweep_pair.num_mkeys',
    'sweep_pair.num_nkeys',
    'sweep_pair.dict_size',
    'explained_variance',
    'parameters_count',
    'accum_num_flops',
    'n_tokens',
    'performance/recovery_from_zero',
    'performance/recovery_from_mean'
]

In [ ]:
mapping = {
    'sweep id': ("1.5B", 1000 * 1e6),
    'sweep id': ("1.5B", 1000 * 1e6),
    'sweep id': ("1.5B", 500 * 1e6),
    'sweep id': ("1.5B", 500 * 1e6),
    'sweep id': ("1.5B", 100 * 1e6),
    'sweep id': ("1.5B", 100 * 1e6),
    
    'sweep id': ("1.5B", 100 * 1e6),
    'sweep id': ("1.5B", 500 * 1e6),
    'sweep id': ("1.5B", 1000 * 1e6),
    
    'sweep id': ("3B", 1000 * 1e6),
    'sweep id': ("3B", 500 * 1e6),
    'sweep id': ("3B", 100 * 1e6),
    
    'sweep id': ("7B", 500 * 1e6),
    'sweep id': ("7B", 100 * 1e6),
    'sweep id': ("1.5B", 1000 * 1e6),

    'sweep id': ("1.4B", 100 * 1e6),
    'sweep id': ("1.4B", 500 * 1e6),
    'sweep id': ("1.4B", 1000 * 1e6),
    'sweep id': ("1.4B", 1000 * 1e6),
    
}

In [ ]:
kronsae = []
topk = []
for i, row in df.iterrows():
    if 'exp51' in row['Name'] and row['Sweep'] in mapping.keys():

        model_size, ref_tokens = mapping[row['Sweep']]

        entry = {
            'Name': row['Name'],
            'H': row['sweep_pair.num_heads'],
            'M': row['sweep_pair.num_mkeys'],
            'N': row['sweep_pair.num_nkeys'],
            'F': row['sweep_pair.dict_size'],
            'Explained Variance': row['explained_variance'],
            'CE_from_zero': row['performance/recovery_from_zero'],
            'CE_from_mean': row['performance/recovery_from_mean'],
            'Parameters': row['parameters_count'],
            'FLOPS': row['accum_num_flops'],
            'Tokens': row['n_tokens'],
            'Reference Tokens': ref_tokens,
            'Model size': model_size
        }
        
        if 'kronsae' in row['Name']:
            entry['Experts'] = entry['H'] * entry['M']
            kronsae.append(entry)
        else:
            topk.append(entry)

kronsae = pd.DataFrame(kronsae)
topk = pd.DataFrame(topk)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.lines import Line2D
import numpy as np

sns.set_style("whitegrid")
fig, axs = plt.subplots(1, 3, figsize=(15, 5), dpi=250, sharey='row')

for i, n_tokens in enumerate(sorted(kronsae['Reference Tokens'].unique(), reverse=False)):
    plot_df = kronsae[kronsae['Reference Tokens'] == n_tokens]
    plot_df = plot_df[plot_df["M"] <= plot_df["N"]]
    ax = axs[i]

    sns.lineplot(
        data=plot_df,
        x='H',  # Heads (powers of 2)
        y='Explained Variance',
        hue='F',
        style='M',
        markers=True, 
        dashes=True,
        palette='tab10',
        markersize=11,
        linewidth=1.5,
        ax=ax,
        legend=i==0
    )

    ax.set_xscale('log', base=2)
    ax.set_xticks(plot_df['H'].unique())
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())

    # Customize plot
    ax.set_xlabel('Trainable Parameters', fontsize=18)
    ax.set_ylabel('Explained Variance', fontsize=18)
    ax.grid(True, alpha=0.3, which='both')

    ax.set_title(f"Token Budget: {n_tokens // 1e6:.0f}M", fontsize=18)
    ax.tick_params(axis='y', which='major', labelsize=14)
    ax.tick_params(axis='x', which='major', labelsize=12)
    ax.tick_params(axis='both', which='minor', labelsize=10)

axs[0].set_xlabel('')
axs[0].set_xlabel('')

handles, labels = axs[0].get_legend_handles_labels()

axs[0].legend(
    handles=handles[1:4] + handles[5:],
    labels=['$F = 2^{15}$', '$F = 2^{16}$', '$F = 2^{17}$', 'm = 2', 'm = 4', 'm = 8'],fontsize=14,
    loc='upper left'
)

axs[0].set_xlabel('')
axs[2].set_xlabel('')

plt.tight_layout()
plt.savefig(f'heads vs explained variance qwen1.5B.pdf')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.lines import Line2D
import numpy as np
from scipy.interpolate import PchipInterpolator
sns.set_style("whitegrid")
fig, axs = plt.subplots(1, 3, figsize=(15, 5), dpi=250, sharey=True)


import numpy as np

def pareto_frontier(xs, ys):
    """
    Extract the 2D Pareto frontier from points (xs, ys).
    Keeps points with strictly increasing y as x increases.
    Returns arrays (pf_x, pf_y) sorted by pf_x ascending.
    """
    pts = sorted(zip(xs, ys), key=lambda p: p[0])
    frontier = []
    best_y = -np.inf
    for x, y in pts:
        if y > best_y:
            frontier.append((x, y))
            best_y = y
    pf_x, pf_y = zip(*frontier)
    return np.array(pf_x), np.array(pf_y)

from scipy.optimize import curve_fit
from scipy.stats import linregress

def smooth_pareto_front(xs, ys, deg=2, num=200, scaler = 1.):
    """
    Compute and return a smooth polynomial approximation of the Pareto front.

    Args:
      xs (array-like):  x-coordinates of all points.
      ys (array-like):  y-coordinates of all points.
      deg (int):        degree of the fitting polynomial.
      num (int):        number of points to sample on the smooth curve.

    Returns:
      x_dense (np.ndarray):  sorted x-values along the front (length=num).
      y_smooth (np.ndarray): fitted y-values p(x_dense) (length=num).
    """
    print(xs, ys)
    pf_x, pf_y = np.array(xs), np.array(ys)
    print(pf_x, pf_y)

    def func(x, a, b, c):
        return a * x**2 + b * x + c 
        
    popt, pcov = curve_fit(func, np.log2(pf_x), pf_y)

    x_dense = np.linspace(pf_x.min() - scaler * pf_x.min(), pf_x.max() + scaler* pf_x.max(), num)
    y_smooth = func(np.log2(x_dense), *popt) 

    return x_dense, y_smooth


for i, n_tokens in enumerate(sorted(kronsae['Reference Tokens'].unique(), reverse=False)):
    plot_df = kronsae[kronsae['Reference Tokens'] == n_tokens]
    plot_df = plot_df[plot_df["M"] <= plot_df["N"]]
    
    ax = axs[i]
    sns.lineplot(
        data=plot_df,
        x='Parameters',  # Heads (powers of 2)
        y='Explained Variance',
        hue='F',
        style='M',
        markers=True,
        dashes=True,
        palette='tab10',
        markersize=8,
        linewidth=2,
        ax=ax,
        legend=i==0
    )

    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())

    ax.set_xlabel('Trainable Parameters', fontsize=18)
    ax.set_ylabel('Explained Variance', fontsize=18)
    ax.grid(True, alpha=0.3)
    

    cplt = sns.color_palette('tab10', 3)
    g_results = sns.scatterplot(topk[topk['Reference Tokens'] == n_tokens], 
                    x = 'Parameters', 
                    y = 'Explained Variance', 
                    hue = 'F', 
                    s = 400, 
                    marker = '*', 
                    palette=cplt, 
                    legend = False, 
                    ax=ax)


    baseline = topk[topk['Reference Tokens'] == n_tokens]
    xs = baseline['Parameters'].values
    ys = baseline['Explained Variance'].values

    
    x_front, y_front = smooth_pareto_front(xs, ys, num=200, scaler=1.9)

    ax.plot(
        x_front, y_front,
        color='black',
        lw=1.1,
        alpha=0.7,
        label='Fitted Pareto front' if i==0 else None
    )

    idx = [grp['Explained Variance'].idxmax() for D_val, grp in plot_df.groupby("F")]
    xs1 = [grp['Parameters'][idx[j]] for j, (D_val, grp) in enumerate(plot_df.groupby("F"))]
    ys1 = [grp['Explained Variance'][idx[j]] for j, (D_val, grp) in enumerate(plot_df.groupby("F"))]
 
    x_front, y_front = smooth_pareto_front(xs1, ys1,  num=200, scaler=1.9)

    all_xs = np.concatenate([baseline['Parameters'].values, plot_df['Parameters'].values])

    ax.set_xlim(all_xs.min() - 0.15 * all_xs.min(), all_xs.max() + 0.15 * all_xs.max())

    ax.plot(
        x_front, y_front,
        color='red',
        lw=1.1,
        alpha=0.7,
        label='Fitted Pareto front' if i==0 else None
    )
    ax.set_title(f"Token Budget: {n_tokens // 1e6:.0f}M", fontsize=18)
    
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.set_xscale("log", base=2)
    ax.set_ylim(0.775, 0.87)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.tick_params(axis='both', which='minor', labelsize=10)
    

axs[0].get_legend().remove()
handles, labels = axs[0].get_legend_handles_labels()
baseline_handles = [
    Line2D([0], [0], 
           marker="*", 
           #color="green", 
           color = "black",
           linestyle="",
           markersize=12, label=f"TopK SAE"),
]

axs[0].legend(
    handles=handles[1:4] + new_handles + baseline_handles,
    labels=['$F = 2^{15}$', '$F = 2^{16}$', '$F = 2^{17}$', 'm = 2', 'm = 4', 'm = 8', 'TopK'], fontsize=14,
    loc='upper left',
)
axs[0].set_xlabel('')
axs[2].set_xlabel('')

plt.tight_layout()
plt.savefig(f'parameters vs explained variance qwen1.5b.pdf')
plt.show()

In [ ]:
new_df = pd.read_csv("comet.csv")

data = []

sae_type_to_name = {
    "topk": "TopK",
    "mul_fractal_topk": "KronSAE",
    "jumprelu": "JumpReLU",
    "mul_fractal_jumprelu": "KronSAE JumpReLU"
}

for i, row in new_df.iterrows():

    ref_tokens = row['Name'].split("_")[-2]

    entry = {
        'Name': row['Name'],
        "Setup": f"{row['dict_size']} / {ref_tokens}",
        'H': row['num_heads'],
        'M': row['num_mkeys'],
        'N': row['num_nkeys'],
        'F': row['dict_size'],
        'Runtime': row['Runtime'],
        'SAE': sae_type_to_name[row['sae_type']],
        'Explained Variance': row['explained_variance'],
        'Parameters': row['parameters_count'],
        'FLOPS': row['accum_num_flops'],
        'Tokens': row['n_tokens_true'],
        "Reference tokens": ref_tokens,
        'Model size': row['model_name'].split('-')[-1],
    }
    data.append(entry)

data = pd.DataFrame(data)

In [ ]:
import colorsys
import numpy as np

fig, axs = plt.subplots(1, 3, figsize=(12, 4), dpi=250, sharex=True, sharey=True)
tab10 = plt.cm.tab10.colors  # Get base colors: [(0.12, 0.47, 0.71, 1), ...]

for i, dict_size in enumerate([2**15, 2**16, 2**17]):
    subset = data[(data['F'] == dict_size) & (data["Model size"] == "1.5B")]
    
    # Process KronSAE data
    kron = subset[(subset["SAE"] == "KronSAE") & (subset["M"] < 16)].copy()
    if not kron.empty:
        kron['M_float'] = kron['M'].astype(float)
        m_vals = sorted(kron['M_float'].unique())
        kron['M'] = kron['M_float'].apply(lambda x: str(int(x)))  # Clean integer strings
        
        # Generate shades: lighter (small M) → darker (large M)
        base_rgb = tab10[i][:3]  # Base color (R,G,B) for this dict size
        lightness_vals = np.linspace(0.8, 0.4, len(m_vals))  # Light to dark
        kron_colors = []
        for l_val in lightness_vals:
            h, l, s = colorsys.rgb_to_hls(*base_rgb)
            new_rgb = colorsys.hls_to_rgb(h, l_val, s)
            kron_colors.append(new_rgb)
    else:
        m_vals = []
        kron_colors = []
    
    # Process TopK (single bar)
    topk = subset[subset["SAE"] == "TopK"].copy()
    topk['M'] = 'TopK'
    topk_color = (0.267, 0.267, 0.267)  # #444444 (dark gray)
    
    # Combine datasets
    combined = pd.concat([kron, topk])
    m_order = [str(int(m)) for m in m_vals] + ['TopK']  # Numeric sort for KronSAE
    palette = kron_colors + [topk_color]  # KronSAE shades + TopK color
    
    # Plot with custom palette
    sns.barplot(
        combined,
        x="Reference tokens",
        y="Explained Variance",
        hue="M",
        estimator="max",
        ax=axs[i],
        legend=i==0,
        # order="500m, 1000m",
        hue_order=m_order,
        palette=palette,
        order=["500m", "1000m"],
        dodge=True
    )
    
    axs[i].set_ylim(0.76, 0.9)
    axs[i].set_title(f"Dictionary: {dict_size}")
    axs[i].grid(alpha=0.35, axis='y', linestyle='--')

    if i != 1:
        axs[i].set_xlabel("")

# Customize legend (last subplot only)
handles, labels = axs[0].get_legend_handles_labels()
axs[0].legend(
    handles=[handles[0], handles[-2], handles[-1]],  # Smallest M + TopK
    labels=[f"m={labels[0]}", f"m={labels[-2]}", labels[-1]],
    loc="upper left",
    frameon=True,
    framealpha=0.9
)

plt.tight_layout()
plt.savefig("maximum_performance.pdf")
plt.show()